# GörEndir — YouTube Video & Subtitle Downloader

This notebook demonstrates how to use **GörEndir** on Google Colab with Google Drive storage.

## Features
- Download single videos or entire playlists
- Multi-language subtitle extraction (SRT + TXT)
- `Dict[str, int]` input format: `{"url": start_number}` to skip and number from a specific video
- `reverse_download=True`: reverse playlist order (last video → #1)
- `skip_download=True`: download only subtitles, skip video file
- `playlist_end=N`: limit number of videos to download
- tqdm progress bar for every download
- Automatic VTT → clean SRT conversion

## ⚠️ Prerequisites (Important!)
YouTube now requires:
1. **Deno** (JavaScript runtime) + **remote components** — installed/enabled in Steps 1-2
2. **Cookies** — exported from your browser and uploaded to Colab (Step 4)

Without these, you will get errors like:
- `No supported JavaScript runtime could be found`
- `n challenge solving failed`
- `Sign in to confirm you're not a bot`

## Step 1: Install GörEndir

In [ ]:
# Install GörEndir
!pip install -q --upgrade --force-reinstall git+https://github.com/MammadTavakoli/gorendir.git

## Step 2: Install Deno (JavaScript Runtime)

yt-dlp needs Deno to solve YouTube's JavaScript challenges.
Without it, you get: `No supported JavaScript runtime could be found` and `n challenge solving failed`

In [ ]:
# Install Deno
!curl -fsSL https://deno.land/install.sh | sh
import os
os.environ["PATH"] = os.path.expanduser("~/.deno/bin") + ":" + os.environ["PATH"]
!deno --version

## Step 3: Update yt-dlp

Make sure you have the latest yt-dlp version with EJS (External JS) support.

In [ ]:
# Update yt-dlp to latest version
!pip install -q --upgrade yt-dlp
!yt-dlp --version

## Step 4: Mount Google Drive

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Step 5: Upload YouTube Cookies

YouTube requires authentication to avoid bot detection.
Without cookies, you will get: **"Sign in to confirm you're not a bot"**

### How to get cookies.txt:
1. Install a browser extension to export cookies:
   - **Chrome**: [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc)
   - **Firefox**: [cookies.txt](https://addons.mozilla.org/en-US/firefox/addon/cookies-txt/)
2. Log into YouTube in your browser
3. Click the extension and export cookies for youtube.com → save as `cookies.txt`
4. Upload the file using the cell below

Alternatively, place `cookies.txt` in your Google Drive and set the path directly.

In [ ]:
# Option A: Upload cookies.txt directly
from google.colab import files
import os

uploaded = files.upload()
cookies_filename = list(uploaded.keys())[0] if uploaded else None

if cookies_filename:
    os.rename(cookies_filename, "/content/cookies.txt")
    cookies_path = "/content/cookies.txt"
    print(f"✅ Cookies uploaded: {cookies_path}")
else:
    cookies_path = None
    print("⚠️ No cookies uploaded. Downloads may fail with bot detection error.")

In [ ]:
# Option B: Use cookies from Google Drive (uncomment and set path)
# cookies_path = "/content/drive/MyDrive/cookies.txt"
# print(f"Using cookies from: {cookies_path}")

## Step 6: Setup

In [ ]:
# Import GörEndir
from gorendir import YouTubeDownloader

In [ ]:
# Define the save location (Google Drive or local Colab storage)
save_path = "/content/drive/MyDrive/YouTube/"

## Step 7: Helper Function

The `download_videos` function wraps `YouTubeDownloader.download_video()`.

### Input Format for `video_urls`

| Format | Example | Description |
|--------|---------|-------------|
| `str` | `"https://youtube.com/watch?v=XXX"` | Single URL |
| `Dict[str, int]` | `{"https://youtube.com/playlist?list=XXX": 8}` | URL with start number |
| `List[Union[str, Dict[str, int]]]` | `["url1", {"url2": 5}]` | Mixed list |

When using `Dict[str, int]`, the integer value means:
- **Skip** the first N-1 videos in download order
- **Number** files starting from that number

In [ ]:
def download_videos(video_urls, save_path, reverse_download=False, skip_download=False, playlist_end=0, cookies_path=None):
    """
    Download videos using GörEndir.

    Args:
        video_urls: URL(s) to download. Can be:
            - str: Single URL
            - Dict[str, int]: {"url": start_number}  e.g. {"https://...playlist?list=XXX": 8}
            - List[Union[str, Dict[str, int]]]: Mixed list of URLs and URL-start_number dicts
        save_path: Directory to save downloads
        reverse_download: If True, reverse playlist order (last video -> #1)
        skip_download: If True, only download subtitles (skip video file)
        playlist_end: If > 0, stop after downloading this many videos
        cookies_path: Path to cookies.txt for YouTube authentication
    """
    downloader = YouTubeDownloader(
        save_directory=save_path,
        max_resolution=1080,
        subtitle_languages=["en", "fa"],
        cookies_path=cookies_path,
    )

    downloader.download_video(
        video_urls=video_urls,
        reverse_download=reverse_download,
        skip_download=skip_download,
        force_download=False,
        yt_dlp_write_subs=True,
        download_subtitles=True,
        playlist_end=playlist_end,
    )

## Step 8: Download Configurations

Define different download scenarios. Each config has:
- `name`: A label for display
- `reverse_download`: Whether to reverse playlist order
- `skip_download`: Whether to skip video (only download subtitles)
- `playlist_end`: Limit number of videos (0 = no limit)
- `urls`: The URL(s) - can be a **list of strings**, a **dict of url→start_number**, or a **mixed list**

In [ ]:
# -- Subtitle-only (reverse order) --
sbtitle_reverse_urls = {
    'name': 'sbtitle_reverse_urls',
    'reverse_download': True,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 8},
    ]
}

In [ ]:
# -- Video + Subtitle (reverse order) --
video_reverse_urls = {
    'name': 'video_reverse_urls',
    'reverse_download': True,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
    ]
}

In [ ]:
# -- Subtitle-only (normal order) --
sbtitle_urls = {
    'name': 'sbtitle_urls',
    'reverse_download': False,
    'skip_download': True,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 3},
    ]
}

In [ ]:
# -- Video + Subtitle (normal order) --
video_urls_config = {
    'name': 'video_urls',
    'reverse_download': False,
    'skip_download': False,
    'playlist_end': 0,
    'urls': [
        # "https://youtube.com/watch?v=YOUR_VIDEO_ID",
        # "https://youtube.com/playlist?list=YOUR_PLAYLIST_ID",
        # {"https://youtube.com/playlist?list=YOUR_PLAYLIST_ID": 5},
    ]
}

## Step 9: Run Downloads

In [ ]:
video_list = [sbtitle_reverse_urls, video_reverse_urls, sbtitle_urls, video_urls_config]

for config in video_list:
    print('*' * 20, config['name'], '*' * 20)
    urls = config['urls']
    if not urls:
        print(f"  No URLs configured for {config['name']}, skipping.")
        continue

    download_videos(
        video_urls=urls,
        save_path=save_path,
        reverse_download=config['reverse_download'],
        skip_download=config['skip_download'],
        playlist_end=config.get('playlist_end', 0),
        cookies_path=cookies_path,
    )

---

## Advanced Usage Examples

In [ ]:
# Example 1: Single video
# download_videos("https://youtube.com/watch?v=dQw4w9WgXcQ", save_path, cookies_path=cookies_path)

In [ ]:
# Example 2: Dict format — skip first 7, start numbering from 8
# download_videos({"https://youtube.com/playlist?list=PLxxxxxxx": 8}, save_path, cookies_path=cookies_path)

In [ ]:
# Example 3: Mixed list
# download_videos([
#     "https://youtube.com/watch?v=abc123",
#     {"https://youtube.com/playlist?list=PLxxxx": 5},
# ], save_path, cookies_path=cookies_path)

In [ ]:
# Example 4: Reverse + start number
# download_videos(
#     {"https://youtube.com/playlist?list=PLxxxx": 8},
#     save_path,
#     reverse_download=True,
#     cookies_path=cookies_path
# )

In [ ]:
# Example 5: Limit to 10 videos
# download_videos(
#     "https://youtube.com/playlist?list=PLxxxx",
#     save_path,
#     playlist_end=10,
#     cookies_path=cookies_path
# )

In [ ]:
# Example 6: YouTubeDownloader directly with all options
# downloader = YouTubeDownloader(
#     save_directory=save_path,
#     max_resolution=720,
#     subtitle_languages=["en", "fa", "tr"],
#     retry_attempts=5,
#     cookies_path=cookies_path,
# )
# downloader.download_video(
#     video_urls={"https://youtube.com/playlist?list=PLxxxx": 3},
#     reverse_download=False,
#     skip_download=False,
#     force_download=True,
#     yt_dlp_write_subs=True,
#     download_subtitles=True,
#     playlist_end=0,
# )